# Lab 06: Building a RAG over Digital-Forensics Reports

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danjethh/sectratek-AI-Lab/blob/main/lab06_llm_rag_case_reports/03_lab_notebook.ipynb)

**Course:** AI for Cybersecurity and Digital Forensics  
**Instructor:** Daniel Kwaku Ntiamoah Addai, Mobile Threat Research Lead, SectraTEK Lab  
**Runs in:** Google Colab or any hosted Jupyter. Nothing is installed on your own computer.

---

Over time an examiner writes many case reports. Months later, someone asks: *"Which of our cases
involved stolen passwords?"* Reading every report again takes hours. If you simply ask an AI model,
it has never seen your reports, so it may refuse or, worse, make up a confident answer.

**RAG** (short for Retrieval-Augmented Generation) fixes this. In plain words it means **"look it
up first, then answer"**: the notebook finds the passages of the case reports most related to your
question and gives only those to the AI. The AI must answer from those passages and say which report
each fact came from, so you can check it.

In this lab you build a small RAG system **by hand**, one step at a time:

1. **Split** the reports into short passages (Step A.1).
2. **Label each passage by meaning**: turn it into numbers a computer can compare (Step A.2).
3. **Find** the passages most related to a question (Step A.3).
4. **Answer** from only those passages, naming the report behind each claim (Step A.4).

Then Part B compares the AI with and without the reports, and Part C lets you ask your own
questions.

The forensic point is the same one that runs through this whole course: the answer must stay inside
what the evidence (here, the reports) actually says. The AI assists; the examiner checks and decides.

## Part 0: One-Time Setup

Run these three steps in order. Together they:

1. install three helper libraries (Step 0.1),
2. connect the notebook to a free AI model that runs online at a company called Groq, used for
   writing the answers (Step 0.2),
3. save the six case reports next to the notebook (Step 0.3).

Everything happens inside this online session. Nothing is installed on your own computer.

### Step 0.1: Install the libraries

**Purpose:** install the three helper libraries this lab uses.

**Input:** nothing from you. This is the first step.

**How it works:** `%pip install` downloads them into this online session only:

- `sentence-transformers` turns each passage into numbers that stand for its meaning (Step A.2).
- `numpy` does fast maths on those numbers, to score how related a passage is to a question
  (Step A.3).
- `openai` sends messages to the AI model; Groq accepts the same format (Steps 0.2 and A.4).

**Output:** an install log that takes a minute or two. Warnings are normal. Step 0.2 uses the
`openai` library to connect to the AI model.

In [ ]:
# Install the three helper libraries this lab needs (only inside this online session).
# sentence-transformers: turns text into lists of numbers that stand for its meaning (Step A.2)
# numpy: fast maths on those numbers, used to score how close two meanings are (Step A.3)
# openai: sends the final question to the AI model (Groq accepts the same format)
%pip install -q sentence-transformers numpy openai

### Step 0.2: Connect to a free hosted model (Groq)

**Purpose:** connect the notebook to a free AI model, so that Step A.4 can write answers in plain
English. The search steps (A.1 to A.3) do not use this model at all.

**Input:** the `openai` library from Step 0.1, plus a free **API key** (a private password that
lets the notebook use the AI model). To get one:

1. Open **https://console.groq.com/keys**, sign in, and create a key.
2. Run the cell and paste the key when asked. It stays hidden as you type and is **not saved**
   into the notebook.

**How it works:** the cell builds a **client** (the notebook's connection to Groq), picks the first
AI model on its list that your key can use, and sends a one-word test message. It also adds a small
safety wrapper: it keeps each reply short enough for the free plan, waits and tries again if Groq
says you are sending too fast, and asks again if the model replies in the wrong form.

**Output:** a line like `Model OK: openai/gpt-oss-120b -> ready`. The connection is saved as
`client` and the model name as `MODEL`. Steps A.4, B.1, and B.2 use both.

In [ ]:
# Connect to a free AI model hosted by Groq, then send a test message to prove it works.
import os        # os: lets the notebook read and store settings, such as your key
import getpass   # getpass: lets you type your secret key without it showing on screen
from openai import OpenAI  # OpenAI: sends your question to the AI model (Groq accepts the same format)

# Ask for your key only if it is not already stored in this session.
if not os.environ.get('GROQ_API_KEY'):
    os.environ['GROQ_API_KEY'] = getpass.getpass('Paste your Groq API key (hidden): ').strip()

# The web address of Groq's service.
BASE_URL = 'https://api.groq.com/openai/v1'
# A list of free AI models to try, best first. Groq retires models from time to time, so this
# cell uses the first one your key can reach and prints its name. Edit the list to try another.
PREFERRED = ['openai/gpt-oss-120b', 'qwen/qwen3.8-27b']

# Build the client: the notebook's connection to Groq.
# max_retries=8: the free plan allows only so much text per minute (about 8,000 "tokens"; a token
# is a small piece of a word). If we go over, Groq says "try again in a few seconds", and the
# client waits and retries instead of stopping with an error.
client = OpenAI(base_url=BASE_URL, api_key=os.environ['GROQ_API_KEY'], max_retries=8)

# A small safety wrapper. Every message this notebook sends to the AI goes through it.
# It does two jobs:
# 1. It caps how long each reply can be: 1,500 tokens, or 900 for Qwen, whose free plan allows
#    only 1,000 reply tokens a minute. Groq counts this cap against the per-minute budget.
#    (A single call can still choose its own cap.)
# 2. gpt-oss sometimes tries to use a built-in feature instead of writing text, and Groq rejects
#    that with the error "model called a tool". The wrapper then asks again (up to 3 tries in
#    all) with a reminder to reply in plain text.
_create = client.chat.completions.create
def _create_within_limits(*args, **kw):
    kw.setdefault('max_tokens', 900 if 'qwen' in str(kw.get('model')) else 1500)
    for attempt in range(3):
        try:
            return _create(*args, **kw)
        except Exception as err:
            if 'called a tool' not in str(err) or attempt == 2:
                raise
            kw['messages'] = list(kw.get('messages', [])) + [
                {'role': 'user', 'content': 'Reply in plain text only; do not call functions.'}]
client.chat.completions.create = _create_within_limits   # from now on, every call uses the wrapper

# Ask Groq which models your key can use, and pick the first one from our list.
available = [m.id for m in client.models.list().data]
MODEL = next((m for m in PREFERRED if m in available), None)
# If none of our preferred models is available, print the ones that are, and stop here.
if MODEL is None:
    print('None of PREFERRED is on your key right now. Models your key can use:')
    for m in available:
        print('  -', m)
    raise SystemExit('Put one of these chat models first in PREFERRED and re-run this cell.')

# Send a one-word test message to prove the connection works.
probe = client.chat.completions.create(
    model=MODEL, messages=[{'role': 'user', 'content': 'Reply with one word: ready'}])
print('Model OK:', MODEL, '->', (probe.choices[0].message.content or '').strip())

### Step 0.3: Save the case reports

**Purpose:** save six short case reports into a `reports/` folder. These reports are the library
your RAG system will search.

**Input:** the report texts, written out in full inside the code cell below. They are
**synthetic**: fictional reports written for teaching, with no real people, devices, or
indicators.

| Report file | What the case is about |
|---|---|
| `DF-2026-017_mobile_exfiltration.md` | An Android phone and a possible customer-data leak. |
| `DF-2026-021_windows_ransomware.md` | A Windows workstation encrypted by ransomware. |
| `DF-2026-025_memory_credential_theft.md` | Memory (RAM) analysis of a laptop and stolen credentials. |
| `DF-2026-030_ios_stalkerware.md` | An iPhone checked for stalkerware (hidden tracking software). |
| `DF-2026-034_disk_deleted_recovery.md` | A USB drive image and recovery of deleted files. |
| `DF-2026-039_network_intrusion.md` | A network intrusion and lateral movement. |

**How it works:** the cell keeps each report's text in a dictionary called `REPORTS` (a list of
file name -> report text pairs). It makes the `reports/` folder and writes each report into it,
replacing any older copy.

**Output:** `Saved 6 reports to ...` followed by the six file names. Step A.1 reads these files.

In [ ]:
# Write the six case reports into a folder called reports/, so the notebook has its own library.
from pathlib import Path  # Path: a simple way to point at files and folders by name

# REPORTS is a dictionary (a list of name -> value pairs): file name -> the report's full text.
REPORTS = {'DF-2026-017_mobile_exfiltration.md': '# Case DF-2026-017: Suspected Customer-Data Exfiltration\n\n*Synthetic teaching report. All names, systems, and indicators are fictional.*\n\nExaminer: Analyst B, SectraTEK Lab\nDevice: Google Pixel 7 (Android 14)\nAcquisition: full file system, Cellebrite UFED 7.64\nIncident window (UTC): 2026-02-15 01:00 to 01:15\n\n## Summary\nA company-issued Android phone was reviewed after a possible leak of the file\ncustomers_q1.csv before an employee\'s departure. The artifacts show the file being staged in\nTelegram storage, outbound Telegram and Gmail messages, related network activity, and deletion\nof the original file from Downloads.\n\n## Key findings\n- 01:05:22 - customers_q1.csv created in /Download.\n- 01:07:41 - the same file hash copied into Telegram\'s Documents folder.\n- 01:07:58 - outbound Telegram message: "sending that sheet now".\n- 01:09:44 - outbound Gmail message: "see attached export".\n- 01:12:04 - customers_q1.csv deleted from Downloads.\n\n## Indicators\n- SHA-256 of customers_q1.csv: 531ec9ebf8f4492754f16b1437f1824995c8bf4ee2f6be74d4f2f13fdf3f2ab5\n- Network endpoints: api.telegram.org, smtp.gmail.com\n\n## Conclusion\nThe evidence supports staging and attempted external sharing of the file through Telegram and\nGmail. It does not, on its own, confirm that the file was delivered to or received by any\nrecipient. Recommend correlation with recipient-side or provider records.\n', 'DF-2026-021_windows_ransomware.md': '# Case DF-2026-021: Windows Ransomware Incident\n\n*Synthetic teaching report. All names, systems, and indicators are fictional.*\n\nExaminer: Analyst C, SectraTEK Lab\nDevice: Dell OptiPlex workstation (Windows 11)\nAcquisition: E01 disk image, FTK Imager\n\n## Summary\nA finance workstation was encrypted by ransomware. Hundreds of documents were renamed with a\n.lxk extension and a ransom note, README_RECOVER.txt, was dropped in every affected folder.\n\n## Key findings\n- Initial access via a malicious macro in an email attachment, invoice_8841.xlsm.\n- The macro downloaded a loader from a staging host and executed it from %TEMP%.\n- Volume Shadow Copies were deleted with vssadmin to prevent recovery.\n- Encryption began 2026-03-04 22:11 UTC and ran for roughly 40 minutes.\n\n## Indicators\n- Ransom note filename: README_RECOVER.txt\n- Encrypted extension: .lxk\n- Bitcoin address in note: 1FakeRansomAddr00000000000000000000\n- Loader SHA-256: 9a1d4c0b7e2f5a6c8d3b1e0f4a7c9d2b6e5f8a1c3d7b2e9f0a4c6d8b1e3f5a7c\n- Staging host: updates.fake-cdn-host.example\n\n## Conclusion\nThe incident is consistent with a macro-delivered ransomware loader that disabled local\nrecovery before encrypting user documents. Behavior resembles common LockBit-style tooling, but\nattribution is not confirmed. No evidence of data exfiltration was found in the available logs.\n', 'DF-2026-025_memory_credential_theft.md': '# Case DF-2026-025: Memory Forensics and Credential Theft\n\n*Synthetic teaching report. All names, systems, and indicators are fictional.*\n\nExaminer: Analyst B, SectraTEK Lab\nDevice: HP EliteBook laptop (Windows 10)\nAcquisition: live RAM capture (16 GB), analyzed with Volatility 3\n\n## Summary\nA laptop suspected of credential theft was captured while powered on. Memory analysis found a\ncredential-dumping tool resident in memory and signs that domain credentials were harvested.\n\n## Key findings\n- A process named svch0st.exe (note the zero) was running from a user profile, not System32.\n- Strings consistent with Mimikatz (sekurlsa, logonpasswords) were present in that process.\n- The LSASS process had an unusual handle opened by svch0st.exe, consistent with a memory read.\n- A scheduled task re-launched the tool at logon for persistence.\n\n## Indicators\n- Malicious process: svch0st.exe (PID 4821)\n- Parent process: powershell.exe launched from a macro\n- Persistence: scheduled task "WindowsUpdateHelper"\n- Observed technique: LSASS memory access for credential extraction\n\n## Conclusion\nThe evidence supports in-memory credential dumping of domain accounts via a Mimikatz-like tool.\nHarvested credentials may enable lateral movement. Recommend a domain-wide password reset and\nreview of authentication logs for pass-the-hash activity.\n', 'DF-2026-030_ios_stalkerware.md': '# Case DF-2026-030: iOS Stalkerware Assessment\n\n*Synthetic teaching report. All names, systems, and indicators are fictional.*\n\nExaminer: Analyst A, SectraTEK Lab\nDevice: Apple iPhone 13 (iOS 17)\nAcquisition: logical + sysdiagnose, analyzed with MVT and manual review\n\n## Summary\nA personal iPhone was examined after the owner reported being tracked. The device showed signs\nconsistent with a commercial monitoring app (stalkerware), including a configuration profile and\nperiodic background network beaconing.\n\n## Key findings\n- A configuration profile granted VPN and web-filter permissions, installed outside MDM.\n- Periodic HTTPS beacons every 15 minutes to a monitoring back-end during charging hours.\n- Elevated background location and microphone access by a non-App-Store helper process.\n- The app icon was hidden from the home screen.\n\n## Indicators\n- Beacon domain: sync.fake-familytrack.app\n- Beacon interval: approximately every 15 minutes\n- Profile display name: "Device Services"\n- Classification: dual-use monitoring / stalkerware (web-verified vendor)\n\n## Conclusion\nFindings are consistent with installed stalkerware performing location and audio monitoring. On\nan unsupervised device, profile payload contents are limited, so some detections are\ncapability-dependent. Recommend removing the profile and rotating the owner\'s account passwords.\n', 'DF-2026-034_disk_deleted_recovery.md': '# Case DF-2026-034: Disk Image and Deleted-File Recovery\n\n*Synthetic teaching report. All names, systems, and indicators are fictional.*\n\nExaminer: Analyst C, SectraTEK Lab\nDevice: external USB drive (exFAT), 128 GB\nAcquisition: raw dd image, verified by SHA-256\n\n## Summary\nAn external drive suspected of holding deleted evidence was imaged and carved. Several images\nand a spreadsheet were recovered from unallocated space, and signs of anti-forensic wiping were\nobserved.\n\n## Key findings\n- File carving recovered 42 JPEG images and one XLSX from unallocated clusters.\n- Filesystem metadata for the originals had been overwritten, but file content survived.\n- A wiping utility had been run, zeroing the directory entries but not all data clusters.\n- Timestamps on recovered content predated the reported wipe by several days.\n\n## Indicators\n- Recovered artifacts: 42 x JPEG, 1 x XLSX (from unallocated space)\n- Anti-forensic tool signature: bulk directory-entry zeroing\n- Verification: image SHA-256 matched the acquisition log\n\n## Conclusion\nThe evidence supports an attempt to destroy files by wiping directory entries, which did not\nfully remove the underlying data. Carving recovered usable content. The recovered spreadsheet\nshould be compared by hash against known case files.\n', 'DF-2026-039_network_intrusion.md': "# Case DF-2026-039: Network Intrusion and Lateral Movement\n\n*Synthetic teaching report. All names, systems, and indicators are fictional.*\n\nExaminer: Analyst A, SectraTEK Lab\nScope: corporate network, endpoint and PCAP review\nAcquisition: endpoint logs plus full packet capture (Zeek + Suricata)\n\n## Summary\nAn intrusion began with a phishing email and progressed to lateral movement and command-and-\ncontrol over DNS. Two servers were accessed before containment.\n\n## Key findings\n- Initial access: a phishing link harvested a user's credentials on a lookalike login page.\n- The attacker used valid credentials over VPN during off-hours.\n- Lateral movement used SMB and remote service creation between workstations.\n- Command-and-control used DNS tunnelling with long, encoded subdomains.\n\n## Indicators\n- Phishing domain: login.fake-sso-portal.example\n- C2 technique: DNS tunnelling (encoded TXT queries)\n- C2 domain: c2.fake-telemetry-dns.example\n- Lateral movement: SMB plus remote service creation (technique T1021)\n\n## Conclusion\nThe evidence supports a credential-phishing intrusion followed by hands-on-keyboard lateral\nmovement and DNS-based C2. No ransomware or encryption was observed. Recommend credential\nresets, DNS egress monitoring, and review of the two accessed servers for data staging.\n"}

# Make the reports/ folder (no error if it already exists), then write every report into it.
reports_dir = Path('reports'); reports_dir.mkdir(exist_ok=True)
for name, text in REPORTS.items():
    (reports_dir / name).write_text(text)
# Confirm what was saved, and where.
print('Saved', len(REPORTS), 'reports to', reports_dir.resolve())
for name in REPORTS:
    print('  -', name)

## Part A: Build the RAG by Hand

You will now build the four RAG steps one at a time, so you can see what each one does before you
start asking questions:

- **Step A.1** splits the reports into passages.
- **Step A.2** turns each passage into numbers that stand for its meaning.
- **Step A.3** finds the passages most related to a question.
- **Step A.4** gives only those passages to the AI and asks for an answer with sources.

### Step A.1: Load the reports and split them into passages (chunking)

**Purpose:** cut each report into short passages. Later, the notebook can then pick out just the
parts that matter for a question, instead of handing the AI whole reports. (These passages are
often called **chunks**, and this step is called **chunking**.)

**Input:** the six report files in `reports/` from Step 0.3.

**How it works:** a helper function, `split_into_chunks`, splits a report at its blank lines (its
paragraph breaks). It then joins neighbouring paragraphs together until a passage reaches about 90
words; a single paragraph longer than that stays whole. Each passage is stored together with the
name of the report it came from, so any answer can name its source later.

**Output:** a line such as `Made N passages from 6 reports.`, then the first 300 characters of
one example passage. All the passages are saved in a list called `passages`; Step A.2 uses it.

In [ ]:
# Read every report and cut it into short passages, each labelled with its source report.
from pathlib import Path

reports_dir = Path('reports')   # the folder written in Step 0.3

# A function (a named, reusable set of steps) that cuts one report's text into passages.
def split_into_chunks(text, max_words=90):
    # Split the text wherever there is a blank line. That gives us its paragraphs.
    paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]
    chunks, current = [], ''   # chunks: finished passages; current: the passage being built
    # A loop: look at each paragraph in turn.
    for p in paragraphs:
        # If adding this paragraph would make the passage longer than max_words, close the
        # current passage and start a new one with this paragraph.
        if len((current + ' ' + p).split()) > max_words and current:
            chunks.append(current.strip()); current = p
        # Otherwise, add this paragraph to the passage being built.
        else:
            current = (current + '\n\n' + p).strip()
    # Keep the last passage too.
    if current:
        chunks.append(current.strip())
    return chunks

# Build the full list of passages. Each one is a small dictionary (name -> value pairs) that keeps
# the passage text AND the report file it came from, so answers can name their source.
passages = []
for path in sorted(reports_dir.glob('*.md')):   # every .md report in reports/, in name order
    for chunk in split_into_chunks(path.read_text()):
        passages.append({'source': path.name, 'text': chunk})

# Show how many passages we made, then the start of one example (the second passage in the list).
print('Made', len(passages), 'passages from', len(list(reports_dir.glob('*.md'))), 'reports.')
print('\nExample passage (from', passages[1]['source'] + '):\n')
print(passages[1]['text'][:300], '...')

### Step A.2: Turn each passage into an embedding (a vector of numbers)

**Purpose:** give each passage a numeric "fingerprint" of its meaning. A computer cannot compare
the meaning of words directly, but it can compare numbers, so this lets it tell which passages are
about the same thing as a question.

**Input:** the `passages` list from Step A.1.

**How it works:** the cell downloads a small, free helper model called `all-MiniLM-L6-v2` (about
90 MB) that runs right inside this session. It is **not** the chat AI from Step 0.2: its only job
is to turn text into numbers. It reads each passage and produces a list of 384 numbers that stands
for what the passage is about. That list of numbers is called an **embedding** (some people also
call it a **vector**). You do not need the maths. Just remember: passages about similar things get
similar numbers.

**Output:** a line such as `Embedding table shape: (N, 384) (passages, numbers per passage)`. It
means a table with one row per passage and 384 numbers in each row. The table is saved as
`passage_vectors` and the helper model as `embedder`; Step A.3 uses both.

In [ ]:
# Load a small helper model and turn every passage into a list of numbers (an embedding).
from sentence_transformers import SentenceTransformer  # SentenceTransformer: turns text into numbers that stand for its meaning
import numpy as np  # numpy: stores those numbers as a table and compares them quickly

# Download the small helper model once (about 90 MB). It runs on the free CPU in Colab.
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Turn every passage into its list of numbers. normalize_embeddings=True puts all the lists on
# the same scale, which keeps the closeness score in Step A.3 simple to read.
passage_vectors = embedder.encode([p['text'] for p in passages], normalize_embeddings=True)
# Store them as one table: one row per passage, 384 numbers per row.
passage_vectors = np.array(passage_vectors)
print('Embedding table shape:', passage_vectors.shape, '(passages, numbers per passage)')

### Step A.3: Retrieve the passages closest to a question

**Purpose:** build the search. Given a question, find the passages most related to it. (This step
is called **retrieval**.)

**Input:** the `embedder` and the `passage_vectors` table from Step A.2, and the `passages` list
from Step A.1.

**How it works:** a function called `retrieve(question)` turns the question into numbers in the
same way as the passages. It then gives every passage a **closeness score**: how similar its
meaning is to the question (usually between 0 and 1; higher means more related). The maths name
for this score is **cosine similarity**. The function returns the top few passages (4 unless you
ask for a different number), each with its score and its source report. The cell then tries it
on an example question about stolen credentials and asks for the top 3.

**Output:** three results, each showing `[score] report-file-name` and the start of the passage.
Ask yourself: are these the reports you would have picked? The `retrieve` function is used inside
Step A.4.

In [ ]:
# Build the search function: it finds the passages most related to a question.
def retrieve(question, k=4):   # k: how many passages to return (4 unless you ask for another number)
    # Turn the question into numbers, the same way the passages were turned into numbers in Step A.2.
    q_vector = embedder.encode([question], normalize_embeddings=True)[0]
    # Score every passage against the question in one go (the @ sign does the comparison).
    # Higher score = closer in meaning. The maths is called cosine similarity; you do not need it.
    scores = passage_vectors @ q_vector
    # Sort the scores from highest to lowest and keep the positions of the top k passages.
    top = np.argsort(scores)[::-1][:k]
    # Return the top passages, each with its score, its source report, and its text.
    return [{'score': float(scores[i]), 'source': passages[i]['source'], 'text': passages[i]['text']} for i in top]

# Try the search with an example question, and print the top 3 results.
example_q = 'Which cases involved stolen or dumped credentials?'
for hit in retrieve(example_q, k=3):
    print(f"[{hit['score']:.2f}] {hit['source']}")   # the score and the report name
    print('   ', hit['text'][:120].replace('\n', ' '), '...\n')   # the start of the passage, on one line

### Step A.4: Generate a grounded, cited answer

**Purpose:** turn the found passages into a written answer an examiner can check. The AI must
answer only from those passages and name the report behind each claim. (An answer built only from
the evidence in front of it is called **grounded**; naming the report is the **citation**.)

**Input:** the `retrieve` function from Step A.3, and `client` and `MODEL` from Step 0.2.

**How it works:** a function called `answer(question)` does the whole "look it up first, then
answer" job:

1. It uses `retrieve` to find the 4 passages most related to the question.
2. It pastes those passages, each labelled with its report file name, into the **prompt** (the
   instructions we give the AI).
3. The prompt says: use ONLY these excerpts, put the report file name in square brackets after
   each claim, and if the answer is not there, say "That is not covered in the reports."
4. It sends the prompt to the AI and returns the AI's reply together with the passages it used.

The cell then asks a question about stolen credentials.

**Output:** the AI's answer, with report names in square brackets such as
`[DF-2026-025_memory_credential_theft.md]`, then a `--- sources retrieved ---` list showing the
passages it was given and their scores. Check: does each cited report really say what the answer
claims? The `answer` function is used again in Step B.2 and Part C.

In [ ]:
# Build the full RAG answer function: find related passages, then ask the AI to answer from them.
def answer(question, k=4):
    # 1. Find the k passages most related to the question (Step A.3).
    hits = retrieve(question, k=k)
    # 2. Join those passages into one block of text, each starting with its report name in brackets.
    context = '\n\n'.join(f"[{h['source']}]\n{h['text']}" for h in hits)
    # 3. Write the instructions for the AI. This text is sent to the AI exactly as written.
    prompt = (
        'You are a digital-forensics assistant. Answer the question using ONLY the report excerpts '
        'below. Cite the report file name in square brackets after each claim. If the answer is not in '
        'the excerpts, say "That is not covered in the reports." Do not use outside knowledge.\n\n'
        f'Report excerpts:\n{context}\n\nQuestion: {question}')
    # 4. Send the instructions to the AI and keep the text of its reply.
    reply = client.chat.completions.create(
        model=MODEL,
        messages=[{'role': 'user', 'content': prompt}],
    ).choices[0].message.content
    # Return the answer AND the passages it was based on, so you can check the sources.
    return reply, hits

# Ask a question, print the answer, then list the passages it was given.
reply, hits = answer('Which cases involved stolen or dumped credentials, and how?')
print(reply)
print('\n--- sources retrieved ---')
for h in hits:
    print(f"  [{h['score']:.2f}] {h['source']}")

## Part B: Why Retrieval Matters

To see what RAG adds, you will ask the **same question two ways**: first to the AI alone, with no
reports (Step B.1), then through your RAG system (Step B.2). Watch how the AI alone may guess
specifics, while the RAG answer stays inside the reports and names them.

### Step B.1: Ask the model with no reports

**Purpose:** see what happens when the AI answers a question about *our* case without the
reports. This is the "before" picture.

**Input:** `client` and `MODEL` from Step 0.2, and a question about case `DF-2026-025`. No
passages are given; `retrieve` is not used.

**How it works:** the question is sent straight to the AI with nothing else. The AI has never
seen our (made-up) case reports, so it can only refuse or guess.

**Output:** an answer with no report names in it. It may politely say it does not know, or it may
invent a tool or a method. Either way, nothing in it is tied to the evidence. Step B.2 asks the
same question with retrieval.

In [ ]:
# Ask the AI about case DF-2026-025 with NO reports, to see what it says on its own.
bare_question = 'In case DF-2026-025, what tool was used to steal credentials, and how did it persist?'
bare = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': bare_question}],
).choices[0].message.content
print(bare)

### Step B.2: Ask the same question with retrieval

**Purpose:** the "after" picture. The same question, but this time the notebook looks it up in
the reports first.

**Input:** the `answer` function from Step A.4, and the same question as Step B.1.

**How it works:** `answer` finds the passages most related to the question, then asks the AI to
answer from only those passages and to name the report it used.

**Output:** a specific answer based on the `DF-2026-025` report, which should name the
credential-stealing tool and how it kept running (its **persistence**), with the report name in
brackets. Below it, the `--- sources retrieved ---` list. Compare it with Step B.1, and check the
details against the report text.

In [ ]:
# The same question, answered by the RAG system you built (look it up first, then answer).
reply, hits = answer('In case DF-2026-025, what tool was used to steal credentials, and how did it persist?')
print(reply)
print('\n--- sources retrieved ---')
for h in hits:
    print(f"  [{h['score']:.2f}] {h['source']}")

## Part C: Ask Your Own Questions

Now it is your turn. Use the `answer(...)` function to ask questions across all six reports, the
kind a lead examiner would ask across many cases at once.

### Step C.1: Ask your own question

**Purpose:** practise using the RAG system the way a lead examiner would.

**Input:** the `answer` function from Step A.4, and a question you write.

**How it works:** change the text inside the quotes on the `my_question` line, then run the cell.
You can run it as many times as you like with different questions.

**Output:** the AI's answer and the `--- sources retrieved ---` list. Use the ideas below, and
keep your best examples for Your Short Write-Up.

In [ ]:
# Your turn. Change the question between the quotes, then re-run this cell.
my_question = 'Which cases showed anti-forensic behaviour, such as deleting backups or wiping files?'
reply, hits = answer(my_question)
print(reply)
print('\n--- sources retrieved ---')
for h in hits:
    print(f"  [{h['score']:.2f}] {h['source']}")

### Good questions to try

- *Which cases involved Android devices?*
- *Summarise every case that mentions ransomware or encryption.*
- *What network indicators (domains or addresses) appear across the reports?*
- *Which cases did Analyst B work on?*
- *Was customer data confirmed delivered in the exfiltration case?* (watch whether it stays
  within what the report says)

Tip: the AI only sees the 4 passages that were found, not all six reports. If an answer looks
incomplete, look at the `--- sources retrieved ---` list: the passage it needed may not have been
picked.

## Your Short Write-Up

**Purpose:** show, in your own words, that you checked the RAG system's answers against the
reports.

**Input:** your answers from Steps A.4, B.1, B.2, and C.1.

**How it works:** in the cell below, replace each `...` text with your own answer (keep the quote
marks around it), then run the cell. It prints your answers back, so they are saved in the
notebook.

**Output:** your three answers, one per line. This is part of what you submit.

Answer these three points:

1. Pick one question. Paste the RAG answer, and confirm the cited report actually supports it.
2. Find a question the reports **cannot** answer. Did the system say so, or did it guess?
3. In one sentence: why does retrieval make the answer more trustworthy than the plain model?

In [ ]:
# Your write-up. Replace each '...' text with your own words (keep the quote marks), then run.
writeup = {
    'q1_grounded_answer': '... your question, the answer, and the citation you checked ...',
    'q2_out_of_scope': '... a question not in the reports, and what the system did ...',
    'q3_why_retrieval_helps': '... one sentence ...',
}
# Print each answer so it is saved with the notebook.
for k, v in writeup.items():
    print(k, '->', v)

## Submit

Save the notebook with the search results, the with/without comparison, and your write-up filled in.

- In Colab: **File -> Download -> Download .ipynb**, or **File -> Save a copy in Drive**.

---

*The reports in this lab are synthetic, written for teaching. The RAG system (split, turn into
numbers, find, answer) is built from scratch with `sentence-transformers` and `numpy`, with Groq
writing the final answer, so you can see every step.*